# Instagram export — a first look at your own data

Notebooks 01 and 02 turned the raw export into an anonymised copy: `anonymised/csv/` holds one tidy table
per kind of event (messages, likes, follows, ads shown…), with every person replaced by a pseudonym. This
notebook is the first analysis of that copy. It asks the questions the course asks about platforms — where
does attention go, how does the social graph grow, how much of what you see is chosen by people you follow
and how much by the recommender, who pays to reach you — and answers each one with a chart drawn from
*your* data.

As before, everything runs inside your browser and nothing is uploaded. The data are pseudonymised: do not
try to work out who `user_0417` is — the point of the exercise is the aggregate pattern, not the person.

Run the cells top to bottom with **Shift+Enter**. Not every export contains every table (a creator account
has `insights_*`, an old account may have no `topics.csv`, a quiet one no `comments.csv`): every section
checks what is there and says so when something is missing, rather than failing.

## 0. Load the data

The tables are in `anonymised/csv/` next to this notebook (notebook 02 wrote them there). If you only have
the `anonymised.zip` you downloaded — on another computer, say — drop it into the file browser on the left
and this cell unzips it. A few helpers are defined here and used throughout: `load()` reads a table and
returns `None` if it is not in your export; `have()` checks that a table exists and has the columns a
section needs; `to_dt()` parses timestamps (Instagram writes them in UTC) and converts them to Rome time,
so that "midnight" means midnight for you.

In [ ]:
import os, glob, zipfile
import pandas as pd
import matplotlib.pyplot as plt

# 0a. Find anonymised/csv (or unzip anonymised.zip), looking in this folder first, then in the whole drive.
def csv_dir_in(folder):
    """The csv/ folder with tables inside an anonymised/ folder (also one level deeper, in case the zip had a top folder)."""
    for cand in [os.path.join(folder, "csv")] + glob.glob(os.path.join(folder, "*", "csv")):
        if os.path.isdir(cand) and glob.glob(os.path.join(cand, "*.csv")):
            return cand
    return None

def find_data():
    bases = [os.getcwd()] + ([r for r, _d, _f in os.walk("/drive")] if os.path.isdir("/drive") else [])
    for base in bases:
        folder, zip_path = os.path.join(base, "anonymised"), os.path.join(base, "anonymised.zip")
        found = csv_dir_in(folder) if os.path.isdir(folder) else None
        if found is None and os.path.exists(zip_path):
            print("Unzipping", zip_path)
            with zipfile.ZipFile(zip_path) as z:
                z.extractall(folder)
            found = csv_dir_in(folder)
        if found:
            return found
    return None

DATA = find_data()
if DATA is None:
    print("No data found: run notebook 02 first, or drag your anonymised.zip into the file browser and re-run this cell.")
else:
    print("Data folder:", DATA)

TZ = "Europe/Rome"

def load(name):
    """Read anonymised/csv/<name>.csv as a DataFrame; None if the table is not in this export or is empty."""
    path = os.path.join(DATA or "", f"{name}.csv")
    if not os.path.exists(path):
        return None
    try:
        df = pd.read_csv(path)
    except Exception as e:                       # empty or malformed file
        print(f"  (could not read {name}.csv: {e})")
        return None
    return df if len(df) else None

def have(df, *cols):
    """True if df exists, has rows, and has all the columns listed."""
    return df is not None and len(df) > 0 and all(c in df.columns for c in cols)

def to_dt(s):
    """Parse ISO timestamps (mixed with/without fractions) as UTC, then convert to Rome time."""
    t = pd.to_datetime(s, format="ISO8601", errors="coerce", utc=True)
    try:
        return t.dt.tz_convert(TZ)
    except Exception:
        return t

def flag(s):
    """Robust boolean from a column that may be read as bool or as the strings 'True'/'False'."""
    return s.astype(str).str.strip().str.lower().isin(["true", "1", "yes"])

def missing(section, *tables):
    print(f"{section}: {', '.join(t + '.csv' for t in tables)} not in your export (or missing the columns needed) - skipping.")

# 0b. List the tables with their row counts.
if DATA:
    rows = []
    for p in sorted(glob.glob(os.path.join(DATA, "*.csv"))):
        name = os.path.basename(p)[:-4]
        df = load(name)
        rows.append((name, 0 if df is None else len(df), "" if df is None else ", ".join(df.columns[:6]) + (" ..." if df is not None and len(df.columns) > 6 else "")))
    tables = pd.DataFrame(rows, columns=["table", "rows", "first columns"]).set_index("table")
    display(tables)

## 1. Eight years in one picture — the lifecycle of a user

Platforms earn from attention, and attention is not constant: a typical account is very active in its first
years, then settles into a pattern — fewer posts, more private messages, more passive consumption. The
export records every action with a date, so the whole history of your account fits in one chart: for each
year, how many things *you* did, by kind. (The platform's side — what it showed you — comes in section 5.)

What to look for: when was your peak? Did public actions (posts, stories, comments) give way to private
ones (messages, likes, saves)? This is the shift from "social network" to "messaging + entertainment feed"
that the platform itself went through in the same years.

In [ ]:
kinds = {   # label -> (table, filter)
    "messages sent": ("messages", lambda d: d[flag(d["is_me"])] if "is_me" in d else d),
    "likes":         ("likes", None),
    "story likes":   ("story_likes", lambda d: d[d["author"].astype(str) != "me"] if "author" in d else d),
    "saves":         ("saved", None),
    "comments":      ("comments", None),
    "posts":         ("posts", None),
    "stories":       ("stories", None),
}
per_year = {}
for label, (table, filt) in kinds.items():
    df = load(table)
    if have(df, "timestamp"):
        if filt is not None:
            df = filt(df)
        per_year[label] = to_dt(df["timestamp"]).dt.year.value_counts()

if not per_year:
    missing("Section 1", *sorted({t for t, _ in kinds.values()}))
else:
    activity = pd.DataFrame(per_year).fillna(0).astype(int).sort_index()
    activity.index = activity.index.astype(int); activity.index.name = "year"
    display(activity)
    plt.figure(figsize=(8, 3.5))
    activity.plot(kind="bar", stacked=True, ax=plt.gca(), width=0.8)
    plt.title("Your own activity per year, by kind")
    plt.xlabel(""); plt.ylabel("actions")
    plt.legend(fontsize=8, ncol=2)
    plt.tight_layout(); plt.show()
    peak = activity.sum(axis=1).idxmax()
    print(f"Peak year: {peak} ({activity.sum(axis=1).max():,} actions). "
          f"Share of private actions (messages, likes, saves) in {peak}: "
          f"{activity.loc[peak, [c for c in ['messages sent','likes','story likes','saves'] if c in activity]].sum() / activity.loc[peak].sum():.0%}; "
          f"in the last year: {activity.iloc[-1][[c for c in ['messages sent','likes','story likes','saves'] if c in activity]].sum() / max(activity.iloc[-1].sum(), 1):.0%}.")

## 2. Attention rhythm — when do you message?

Attention is a scarce input and it has a timetable. Advertisers pay more for some hours than others, the
app schedules notifications to catch you when you are most likely to respond, and the recommender learns
your daily routine. Your messages are the densest time series in the export (one row per message, to the
second), so they show your own rhythm best: by hour of the day and by weekday, in Rome time, split between
what you sent and what you received.

Received messages are other people's rhythm, delivered to you; the gap between the two curves is the time
you spend *answering* rather than *starting*.

In [ ]:
msgs = load("messages")
if not have(msgs, "timestamp", "is_me"):
    missing("Section 2", "messages")
else:
    msgs["t"] = to_dt(msgs["timestamp"])
    msgs["me"] = flag(msgs["is_me"])
    ok = msgs.dropna(subset=["t"])
    by_hour = ok.groupby([ok["t"].dt.hour, "me"]).size().unstack(fill_value=0).reindex(range(24), fill_value=0)
    by_hour.columns = ["received" if not c else "sent" for c in by_hour.columns]
    days = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]
    by_day = ok.groupby([ok["t"].dt.dayofweek, "me"]).size().unstack(fill_value=0).reindex(range(7), fill_value=0)
    by_day.columns = ["received" if not c else "sent" for c in by_day.columns]
    by_day.index = days

    fig, axes = plt.subplots(1, 2, figsize=(8, 3.5), gridspec_kw={"width_ratios": [2, 1]})
    by_hour.plot(kind="bar", stacked=True, ax=axes[0], width=0.9)
    axes[0].set_title(f"Messages by hour of day ({TZ})"); axes[0].set_xlabel("hour"); axes[0].set_ylabel("messages")
    axes[0].set_xticks(range(0, 24, 3)); axes[0].set_xticklabels(range(0, 24, 3), rotation=0)
    by_day.plot(kind="bar", stacked=True, ax=axes[1], width=0.8, legend=False)
    axes[1].set_title("by weekday"); axes[1].set_xlabel("")
    plt.tight_layout(); plt.show()

    tot = by_hour.sum(axis=1)
    night = tot.loc[list(range(0, 7))].sum() / max(tot.sum(), 1)
    print(f"{len(ok):,} messages with a timestamp; busiest hour {tot.idxmax()}:00 ({tot.max() / max(tot.sum(), 1):.0%} of all), "
          f"{night:.0%} between midnight and 7:00; busiest day {by_day.sum(axis=1).idxmax()}; "
          f"weekend share {by_day.loc[['Sat', 'Sun']].sum().sum() / max(by_day.sum().sum(), 1):.0%}.")

## 3. Conversations — with whom, and how reciprocal?

Messaging is where the platform's *network effect* lives: Instagram is hard to leave not because of the
feed but because the people you talk to are there. Attention in conversations is very concentrated — a
handful of threads usually carry most of the messages. For each thread the table below gives the number of
messages, the share you sent, and a **reciprocity** index (messages from the quieter side divided by
messages from the louder side: 1 means perfectly balanced, 0 means one of you does all the talking). The
chart shows the ten busiest threads, stacked sent / received.

If timestamps are present, the last columns measure tempo: the median gap between consecutive messages in
the thread, and the median time you take to reply when the other person wrote last (in hours; it includes
the nights and days in which the conversation was simply dormant, so it measures tempo, not politeness).

In [ ]:
msgs = load("messages")
if not have(msgs, "thread", "is_me"):
    missing("Section 3", "messages")
else:
    msgs["me"] = flag(msgs["is_me"])
    g = msgs.groupby("thread").agg(messages=("me", "size"), sent=("me", "sum"))
    g["received"] = g["messages"] - g["sent"]
    g["share sent"] = (g["sent"] / g["messages"]).round(2)
    g["reciprocity"] = (g[["sent", "received"]].min(axis=1) / g[["sent", "received"]].max(axis=1).replace(0, pd.NA)).astype(float).round(2)

    if "timestamp" in msgs:
        msgs["t"] = to_dt(msgs["timestamp"])
        s = msgs.dropna(subset=["t"]).sort_values(["thread", "t"])
        s["gap_h"] = s.groupby("thread")["t"].diff().dt.total_seconds() / 3600
        s["prev_me"] = s.groupby("thread")["me"].shift()
        reply = s[s["me"] & (s["prev_me"] == False)]          # your message right after one of theirs
        g["median gap (h)"] = s.groupby("thread")["gap_h"].median().round(1)
        g["median reply time (h)"] = reply.groupby("thread")["gap_h"].median().round(1)

    top = g.sort_values("messages", ascending=False).head(10)
    display(top)
    plt.figure(figsize=(8, 3.5))
    top[["sent", "received"]].iloc[::-1].plot(kind="barh", stacked=True, ax=plt.gca())
    plt.title("Ten busiest threads: messages sent and received"); plt.xlabel("messages"); plt.ylabel("")
    plt.tight_layout(); plt.show()

    share_top = top["messages"].sum() / g["messages"].sum()
    print(f"{len(g)} threads; the ten busiest carry {share_top:.0%} of all messages. "
          f"Overall you sent {g['sent'].sum() / g['messages'].sum():.0%} of messages; "
          f"median reciprocity across threads with 10+ messages: {g.loc[g['messages'] >= 10, 'reciprocity'].median():.2f}.")

## 4. The network — followers, following, and who moved first

The social graph is the platform's original asset. Two things grow over time: the accounts you chose to
follow (your *outbound* choices — what you want to see) and the accounts that follow you (*inbound*
attention). Their ratio says something about your role: a follower/following ratio well above 1 is a
creator or a public figure, around 1 a friend among friends, well below 1 a reader.

The export stores the date of each follow, so we can also look at **reciprocity in the graph**: how many
of the accounts you follow follow you back (mutual share), and among mutual ties, who followed first. A
follow-back within days is the platform's "X started following you — follow back?" nudge at work.

In [ ]:
fol = load("followers"); fwg = load("following")
if not (have(fol, "user") and have(fwg, "user")):
    missing("Section 4", "followers", "following")
else:
    followers, following = set(fol["user"].astype(str)), set(fwg["user"].astype(str))
    mutual = followers & following
    print(f"{len(followers):,} followers, {len(following):,} following (ratio {len(followers) / max(len(following), 1):.2f}). "
          f"Mutual: {len(mutual):,} = {len(mutual) / max(len(following), 1):.0%} of the accounts you follow, "
          f"{len(mutual) / max(len(followers), 1):.0%} of your followers.")

    if have(fol, "timestamp") and have(fwg, "timestamp"):
        a = to_dt(fol["timestamp"]).dropna().dt.tz_localize(None).dt.to_period("M").value_counts().sort_index().cumsum()
        b = to_dt(fwg["timestamp"]).dropna().dt.tz_localize(None).dt.to_period("M").value_counts().sort_index().cumsum()
        idx = a.index.union(b.index)
        growth = pd.DataFrame({"followers": a.reindex(idx).ffill().fillna(0), "following": b.reindex(idx).ffill().fillna(0)})
        growth.index = growth.index.to_timestamp()
        plt.figure(figsize=(8, 3.5))
        growth.plot(ax=plt.gca())
        plt.title("Cumulative followers and following over time"); plt.xlabel(""); plt.ylabel("accounts")
        plt.tight_layout(); plt.show()

        # Follow-backs: for mutual ties, who followed first and how long the other took.
        fa = fol.assign(t_in=to_dt(fol["timestamp"]), user=fol["user"].astype(str)).dropna(subset=["t_in"]).drop_duplicates("user")
        fb = fwg.assign(t_out=to_dt(fwg["timestamp"]), user=fwg["user"].astype(str)).dropna(subset=["t_out"]).drop_duplicates("user")
        m = fa.merge(fb, on="user")[["user", "t_in", "t_out"]]
        if len(m):
            m["delay_days"] = (m["t_in"] - m["t_out"]).dt.total_seconds() / 86400   # >0: you followed first
            you_first = m["delay_days"] > 0
            print(f"Among {len(m)} mutual ties with both dates: you followed first in {you_first.mean():.0%} of cases "
                  f"(they followed back after a median {m.loc[you_first, 'delay_days'].median():.0f} days); "
                  f"they followed first in {(~you_first).mean():.0%} (you followed back after a median "
                  f"{(-m.loc[~you_first, 'delay_days']).median():.0f} days). "
                  f"Follow-backs within a week, either direction: {(m['delay_days'].abs() <= 7).mean():.0%}.")
    else:
        print("No follow dates in this export: growth over time cannot be drawn.")

## 5. Consumption vs network — the recommender's share

In its first decade Instagram showed you what the accounts you follow posted. Since 2020 or so a growing
share of the feed, and almost all of Reels and Explore, is picked by a recommender from accounts you do
*not* follow. The export lets you measure this from two angles.

**Your reactions, over the whole history.** Each liked or saved post carries `in_network` — whether its
author is in your followers/following lists. The share of likes and saves on *out-of-network* posts, per
year, is a lower bound on how much of what you engaged with came from recommendation rather than from
your social graph. (It is a lower bound because you may have followed the author afterwards.)

**What the feed showed you, in the last few days.** `ads_viewed`, `posts_viewed`, `videos_watched` and
`stories_viewed` are the platform's own log of impressions. Meta keeps it for only a short window, so this
is a snapshot, not a history — but it is the only direct view of the *supply* side of your attention.

In [ ]:
# 5a. Share of likes / saves on accounts you do not follow, per year.
series = {}
for label, table in [("likes", "likes"), ("saves", "saved")]:
    df = load(table)
    if have(df, "timestamp", "in_network"):
        d = df.assign(year=to_dt(df["timestamp"]).dt.year, out=~flag(df["in_network"])).dropna(subset=["year"])
        if "kind" in d:                         # likes.csv mixes liked posts and liked comments
            d = d[d["kind"].astype(str) == "post"] if (d["kind"].astype(str) == "post").any() else d
        d = d[d["author"].astype(str) != "me"] if "author" in d else d
        n = d.groupby("year").size()
        series[label] = d.groupby("year")["out"].mean()[n >= 10]      # years with fewer than 10 events are noise
if not series:
    missing("Section 5a", "likes", "saved")
else:
    rec_share = pd.DataFrame(series).sort_index()
    rec_share.index = rec_share.index.astype(int)
    display(rec_share.round(2))
    plt.figure(figsize=(8, 3.5))
    rec_share.plot(ax=plt.gca(), marker="o")
    plt.title("Share of your likes / saves on accounts you do not follow (per year)")
    plt.xlabel(""); plt.ylabel("share out-of-network"); plt.ylim(0, 1)
    plt.tight_layout(); plt.show()

In [ ]:
# 5b. What the feed showed you in the last days: impressions by source (followed vs not).
rows = {}
span = {}
for table in ["ads_viewed", "posts_viewed", "videos_watched", "stories_viewed"]:
    df = load(table)
    if have(df, "in_network"):
        inn = flag(df["in_network"])
        rows[table] = {"followed": int(inn.sum()), "not followed": int((~inn).sum())}
        if "timestamp" in df:
            t = to_dt(df["timestamp"]).dropna()
            if len(t):
                span[table] = f"{t.min():%Y-%m-%d} to {t.max():%Y-%m-%d} ({(t.max() - t.min()).days + 1} days)"
if not rows:
    missing("Section 5b", "ads_viewed", "posts_viewed", "videos_watched")
else:
    feed = pd.DataFrame(rows).T
    feed["share not followed"] = (feed["not followed"] / feed.sum(axis=1)).round(2)
    feed["period covered"] = pd.Series(span)
    display(feed)
    plt.figure(figsize=(8, 3.5))
    feed[["followed", "not followed"]].plot(kind="barh", stacked=True, ax=plt.gca())
    plt.title("Impressions logged by the platform, by source"); plt.xlabel("items"); plt.ylabel("")
    plt.tight_layout(); plt.show()
    print("Note: ads are by construction from accounts you do not follow; the interesting rows are posts and videos.")

## 6. Taste — what you like, what you keep, how you comment

The recommender does not need to know *who* you are; it needs to know *what* you respond to. The optional
last step of notebook 02 labelled each liked and saved post with a coarse topic (from its caption and
hashtags) and each of your comments with a tone. These labels are rough — a lexicon or a small local
model, not the platform's own classifier — but they let you see your revealed preferences the way an
advertiser would: as a distribution over categories that drifts over the years.

Likes are cheap and many; saves are rarer and signal intent ("I want to come back to this": a recipe, a
place, a product). Comparing the two distributions tells you which topics you *consume* and which you
*collect*. The tone of your comments is the public face you show other users.

In [ ]:
likes = load("likes"); saved = load("saved"); comments = load("comments")
shown = False

if have(likes, "timestamp", "topic"):
    d = likes.dropna(subset=["topic"]).assign(year=to_dt(likes["timestamp"]).dt.year).dropna(subset=["year"])
    if len(d):
        d["year"] = d["year"].astype(int)
        top_topics = d["topic"].value_counts().drop("other", errors="ignore").head(7).index
        d["topic2"] = d["topic"].where(d["topic"].isin(top_topics), "other")
        tt = d.groupby(["year", "topic2"]).size().unstack(fill_value=0)
        tt = tt[tt.sum(axis=1) >= 10]                                    # years with enough liked posts
        shares = tt.div(tt.sum(axis=1), axis=0)
        plt.figure(figsize=(8, 3.5))
        shares.plot(kind="bar", stacked=True, ax=plt.gca(), width=0.8)
        plt.title("Topics of the posts you liked, share per year"); plt.xlabel(""); plt.ylabel("share")
        plt.legend(fontsize=7, ncol=2, loc="upper left", bbox_to_anchor=(1, 1))
        plt.tight_layout(); plt.show()
        shown = True
elif have(likes):
    print("likes.csv has no topic column: run the last cell of notebook 02 to add it.")

if have(saved, "topic") and saved["topic"].notna().any():
    plt.figure(figsize=(8, 3.5))
    saved["topic"].value_counts().head(12).iloc[::-1].plot(kind="barh", ax=plt.gca())
    plt.title("Topics of the posts you saved"); plt.xlabel("saved posts"); plt.ylabel("")
    plt.tight_layout(); plt.show()
    shown = True

if have(comments, "tone") and comments["tone"].notna().any():
    tone = comments["tone"].value_counts()
    print("Tone of your comments:", ", ".join(f"{k} {v / tone.sum():.0%}" for k, v in tone.items()), f"(n = {tone.sum()})")
    shown = True

if not shown:
    missing("Section 6", "likes", "saved", "comments")

## 7. The advertising side — who pays to reach you

Instagram's revenue is advertising, and the export shows three layers of it.

1. **Advertisers holding a match on you** (`advertisers.csv`). These businesses uploaded a *customer list*
   (your e-mail or phone, from a purchase, a newsletter, a loyalty card) or tracked you on their own site or
   app via the Meta pixel, and Meta matched it to your account. This is targeting from *off-platform* data:
   the advertiser already knew you. The list is undated and often long. `how` says which channel.
2. **Ads actually shown** (`ads_viewed.csv`), in the same few-day window as the feed log: which advertisers,
   and how often. The hashtags of the ad carry its *disclosure*: in Italy and the EU paid content must be
   labelled (`#adv`, `#ad`, `#sponsored`, `#pubblicità`, `#supplied`…) — the creator economy's compliance
   layer.
3. **Accounts the platform suggested you follow** (`suggested_accounts.csv`): growth of the graph driven by
   the platform rather than by you.

In [ ]:
adv = load("advertisers"); ads = load("ads_viewed"); sugg = load("suggested_accounts")
shown = False

if have(adv, "advertiser"):
    print(f"{adv['advertiser'].nunique()} advertisers hold a match on you.")
    if "how" in adv:
        display(adv["how"].value_counts().rename("advertisers").to_frame())
    shown = True

if have(ads, "author"):
    by_adv = ads["author"].value_counts()
    print(f"{len(ads)} ads shown, from {len(by_adv)} advertisers; the top 5 account for {by_adv.head(5).sum() / len(ads):.0%} of impressions.")
    plt.figure(figsize=(8, 3.5))
    by_adv.head(10).iloc[::-1].plot(kind="barh", ax=plt.gca())
    plt.title("Ads shown, by advertiser (top 10)"); plt.xlabel("impressions"); plt.ylabel("")
    plt.tight_layout(); plt.show()
    if "hashtags" in ads:
        disclosure = r"\b(?:adv|ad|ads|sponsored|sponsorizzato|pubblicita|pubblicità|paidpartnership|partnership|supplied|gifted|collab)\b"
        tagged = ads["hashtags"].fillna("").astype(str).str.lower().str.contains(disclosure, regex=True)
        with_tags = ads["hashtags"].notna()
        print(f"Ads with any hashtag: {with_tags.mean():.0%}; with a disclosure hashtag (#adv, #ad, #sponsored...): {tagged.mean():.0%} of all ads, "
              f"{tagged[with_tags].mean() if with_tags.any() else 0:.0%} of ads that have hashtags.")
    if have(adv, "advertiser"):
        overlap = set(ads["author"].astype(str).str.lower()) & set(adv["advertiser"].astype(str).str.lower())
        print(f"Advertisers that both hold a match on you and showed you an ad this week: {len(overlap)}"
              " (names rarely coincide exactly - handles vs company names - so this is a lower bound).")
    shown = True

likes = load("likes")
if have(likes, "hashtags"):
    disclosure = r"\b(?:adv|ad|sponsored|sponsorizzato|pubblicita|pubblicità|paidpartnership|supplied|gifted)\b"
    tagged = likes["hashtags"].fillna("").astype(str).str.lower().str.contains(disclosure, regex=True)
    print(f"Organic posts you liked that carry a disclosure hashtag (influencer ads): {tagged.sum()} of {len(likes)} ({tagged.mean():.1%}).")

if have(sugg, "author"):
    inn = flag(sugg["in_network"]) if "in_network" in sugg else pd.Series(False, index=sugg.index)
    when = ""
    if "timestamp" in sugg:
        t = to_dt(sugg["timestamp"]).dropna()
        if len(t):
            when = f", logged between {t.min():%Y-%m} and {t.max():%Y-%m}"
    note = f"; {inn.mean():.0%} of them are now in your network" if inn.nunique() > 1 else ""
    print(f"{len(sugg)} accounts suggested to you{when}{note}.")
    shown = True

if not shown:
    missing("Section 7", "advertisers", "ads_viewed", "suggested_accounts")

## 8. Data-quality notes — what this dataset can and cannot say

Before you write anything from these charts, know the limits of the source. The table below shows, for every
table with a timestamp, how many rows it has and the period it covers; read it with these points in mind.

- **The viewing history is a few days long.** `ads_viewed`, `posts_viewed`, `videos_watched`,
  `stories_viewed` cover roughly the last week before the export. Likes, saves, messages and follows go
  back to the account's creation. Never compare a count from the first group with one from the second
  without dividing by the period covered.
- **`advertisers.csv` is undated.** It is the state of Meta's matching tables at export time; it does not
  say when a business uploaded your contact, nor whether it ever ran an ad you saw.
- **`in_network` is measured today.** An account counts as "followed" if it is in your lists *now*. A
  2019 like on an account you followed in 2021 is labelled in-network; the recommender share in section 5
  is therefore a lower bound, and the bias grows for older years.
- **Pseudonyms change with every run** of notebook 02 (`user_0417` in your file is not `user_0417` in a
  classmate's, nor in your own next run). Never merge two anonymised exports on pseudonyms. `me` is stable.
- **Captions and messages are metadata only.** Word counts, emoji counts, hashtags, flags; the topic and
  tone labels come from a lexicon or a small local model, not from reading the content. Treat them as noisy.
- **Instagram's own numbers are not raw either.** `insights_*` (creator accounts) have gaps and `--` where
  the platform did not count; `likes.csv` mixes liked posts and liked comments (`kind`); timestamps are UTC
  and the `SHIFT_DAYS` option of notebook 02 may have moved every date by the same unknown offset (hours
  and weekdays survive, calendar dates do not).
- **Absence is not evidence.** A missing table means the export did not contain that file, which happens
  for features never used, for old accounts, and for files Meta has quietly stopped exporting.

In [ ]:
cov = []
for p in sorted(glob.glob(os.path.join(DATA or "", "*.csv"))):
    name = os.path.basename(p)[:-4]
    df = load(name)
    if df is None:
        cov.append((name, 0, None, None, None)); continue
    col = next((c for c in ["timestamp", "first_message"] if c in df.columns), None)
    if col is None:
        cov.append((name, len(df), "undated", "", "")); continue
    t = to_dt(df[col]).dropna()
    if len(t) == 0:
        cov.append((name, len(df), "no valid dates", "", "")); continue
    cov.append((name, len(df), f"{t.min():%Y-%m-%d}", f"{t.max():%Y-%m-%d}", (t.max() - t.min()).days + 1))
coverage = pd.DataFrame(cov, columns=["table", "rows", "first", "last", "days covered"]).set_index("table")
display(coverage)

## Discussion questions

1. **Attention allocation.** Section 1 shows how your actions shifted over the years between public
   (posts, stories, comments) and private (messages, likes, saves). Who captures the value of each kind of
   action — you, your contacts, the platform, advertisers? Why would a platform *want* the shift you see?
2. **Network effects.** Section 3 shows that a few threads carry most of your messages, and section 4 how
   many ties are reciprocal. If your ten busiest threads moved to another app tomorrow, what would keep you
   on Instagram? Is the lock-in in the graph, in the content, or in the habit (section 2)?
3. **Recommender vs social graph.** Section 5 estimates the share of your engagement that came from
   accounts you do not follow, and what the feed showed you last week. Who decides what you see — you, your
   network, or the ranking model? What changed in the platform's incentives when the recommender took over
   from the follow graph, and who gained and lost (users, small creators, large creators, advertisers)?
4. **Targeted advertising and customer lists.** Section 7 lists businesses that uploaded your contact details
   to Meta. You gave your e-mail to a shop, not to Instagram: through which transaction did it end up here,
   who was paid, and is there a market for this match? Would the ads you saw be worth less without it?
5. **Creator economy and disclosure.** Sections 6 and 7 show topics you respond to and how many ads carry a
   `#adv`-type label. If a creator you follow is paid to post, who bears the cost of *not* disclosing — the
   follower, the honest competitor, the platform? Does the hashtag fix the information problem?

## What to hand in

Pick **three charts** from this notebook (yours, or re-drawn with a change you justify) and write a
**one-page memo** (max 500 words): for each chart, one paragraph on what it shows in your data and one on
what it says about the economics of the platform, citing at least one concept from the course. Paste the
charts in the memo; do not attach the data. Everything in the charts is aggregated and pseudonymised — if a
chart would let a reader identify a person, choose another one.